# Advanced Flight Price Prediction
## Beyond Linear Regression: GBDT + Deep Learning + Uncertainty Quantification

**Pipeline:**
1. Enhanced feature engineering (10+ new features over baseline)
2. LightGBM, CatBoost, XGBoost with 5-fold cross-validation
3. Quantile regression for prediction intervals
4. TabNet (attention-based deep learning for tabular data)
5. SHAP explainability
6. Statistical significance tests
7. Error analysis

In [ ]:
# Run once to install packages
# !pip install lightgbm catboost xgboost shap pytorch-tabnet -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import gc
import os
import time
warnings.filterwarnings('ignore')

from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import (
    r2_score, mean_absolute_error,
    mean_absolute_percentage_error, mean_squared_error
)
from sklearn.preprocessing import LabelEncoder, StandardScaler

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor, Pool
import shap

import torch
from pytorch_tabnet.tab_model import TabNetRegressor

from scipy.stats import wilcoxon

SEED = 42
N_FOLDS = 5
TARGET = 'totalFare'

np.random.seed(SEED)
torch.manual_seed(SEED)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
plt.rcParams.update({'figure.figsize': (14, 6), 'font.size': 12})
sns.set_style('whitegrid')
sns.set_palette('husl')

print(f'NumPy {np.__version__} | Pandas {pd.__version__}')
print(f'LightGBM {lgb.__version__} | XGBoost {xgb.__version__}')
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')

## 1. Data Loading

In [ ]:
# Handles Kaggle and local environments
if os.path.exists('/kaggle/input/flight-price-subset/flight_prices.csv'):
    DATA_PATH = '/kaggle/input/flight-price-subset/flight_prices.csv'
    print('Kaggle: subset dataset')
elif os.path.exists('/kaggle/input/datasets/dilwong/flightprices/itineraries.csv'):
    DATA_PATH = '/kaggle/input/datasets/dilwong/flightprices/itineraries.csv'
    print('Kaggle: full dataset')
else:
    DATA_PATH = 'flight_prices.csv'
    print('Local')

df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Shape: {df_raw.shape}')
print(f'Memory: {df_raw.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
df_raw.head(3)

## 2. Enhanced Feature Engineering

New features over the baseline notebook:
- `dayOfWeek`, `isWeekend`, `flightQuarter`
- `bookingWindowBucket` (categorical bins of lead time)
- `departureHour`, `timeOfDay`
- `numStops`, `numAirlines`
- `cabinCode`, `fareClass`
- `routePopularity`, `routeNumAirlines` (market competition proxy)
- `avgSpeed` (distance / duration)
- `isPeakSeason`, `isHolidaySeason`
- `isLowAvailability` (seats <= 3)

In [ ]:
def extract_first(series, delimiter='||'):
    return series.astype(str).str.split(delimiter, regex=False).str[0].str.strip()

def count_pipe_segments(series):
    # number of || occurrences + 1 = number of segments
    return series.astype(str).str.count('\\|\\|') + 1

def parse_departure_hour(series):
    try:
        first_seg = series.astype(str).str.split('||', regex=False).str[0].str.strip()
        return pd.to_datetime(first_seg, format='mixed', errors='coerce').dt.hour
    except Exception:
        return pd.Series(np.nan, index=series.index)

def get_time_of_day(hour):
    if pd.isna(hour):
        return 'Unknown'
    h = int(hour)
    if 5 <= h < 9:   return 'EarlyMorning'
    if 9 <= h < 12:  return 'Morning'
    if 12 <= h < 17: return 'Afternoon'
    if 17 <= h < 21: return 'Evening'
    return 'Night'

def get_booking_window(days):
    if days <= 3:   return 'LastMinute'
    if days <= 7:   return 'OneWeek'
    if days <= 14:  return 'TwoWeeks'
    if days <= 30:  return 'OneMonth'
    if days <= 60:  return 'TwoMonths'
    return 'EarlyBird'

print('Helper functions defined.')

In [ ]:
def engineer_features(df):
    d = df.copy()

    # --- Dates ---
    d['flightDate']  = pd.to_datetime(d['flightDate'])
    d['searchDate']  = pd.to_datetime(d['searchDate'])
    d['daysFrom']    = (d['flightDate'] - d['searchDate']).dt.days
    d['flightMonth'] = d['flightDate'].dt.month
    d['flightMonthName'] = d['flightDate'].dt.month_name()
    d['dayOfWeek']   = d['flightDate'].dt.dayofweek
    d['dayOfWeekName'] = d['flightDate'].dt.day_name()
    d['isWeekend']   = (d['dayOfWeek'] >= 5).astype(int)
    d['flightQuarter'] = d['flightDate'].dt.quarter
    d['bookingWindowBucket'] = d['daysFrom'].apply(get_booking_window)
    d['isPeakSeason']    = d['flightMonth'].isin([6, 7, 8]).astype(int)
    d['isHolidaySeason'] = d['flightMonth'].isin([11, 12]).astype(int)

    # --- Duration ---
    d['travelDurationHours'] = (
        pd.to_timedelta(d['travelDuration']).dt.total_seconds() / 3600
    ).round(2)

    # --- Route ---
    d['route'] = d['startingAirport'] + '_' + d['destinationAirport']
    route_counts = d['route'].value_counts()
    d['routePopularity'] = d['route'].map(route_counts)

    # --- Segments / Stops ---
    d['numSegments'] = count_pipe_segments(d['segmentsDepartureAirportCode'])
    d['numStops']    = d['numSegments'] - 1

    # --- Airlines ---
    d['primaryAirline'] = extract_first(d['segmentsAirlineName'])
    d['numAirlines'] = d['segmentsAirlineName'].astype(str).apply(
        lambda x: len(set(s.strip() for s in x.split('||')))
    )
    route_airlines = (
        d.groupby('route')['primaryAirline']
         .nunique()
         .rename('routeNumAirlines')
    )
    d = d.join(route_airlines, on='route')

    # --- Aircraft ---
    d['primaryAircraft'] = extract_first(d['segmentsEquipmentDescription'])
    mfr_pattern = '(Airbus|Boeing|Embraer|Bombardier|ATR)'
    d['aircraftManufacturer'] = (
        d['primaryAircraft']
          .str.extract(mfr_pattern, expand=False)
          .fillna('Other')
    )

    # --- Cabin / Fare class ---
    if 'segmentsCabinCode' in d.columns:
        d['cabinCode'] = extract_first(d['segmentsCabinCode'])
    else:
        d['cabinCode'] = 'coach'

    if 'fareBasisCode' in d.columns:
        d['fareClass'] = d['fareBasisCode'].str[0].str.upper().fillna('X')
    else:
        d['fareClass'] = 'X'

    # --- Departure time ---
    if 'segmentsDepartureTimeRaw' in d.columns:
        d['departureHour'] = parse_departure_hour(d['segmentsDepartureTimeRaw'])
    else:
        d['departureHour'] = np.nan
    d['timeOfDay'] = d['departureHour'].apply(get_time_of_day)

    # --- Distance imputation (route-wise median then global) ---
    d['totalTravelDistance'] = d['totalTravelDistance'].fillna(
        d.groupby('route')['totalTravelDistance'].transform('median')
    )
    d['totalTravelDistance'] = d['totalTravelDistance'].fillna(
        d['totalTravelDistance'].median()
    )

    # --- Speed proxy ---
    d['avgSpeed'] = (d['totalTravelDistance'] / d['travelDurationHours']).round(1)
    d['avgSpeed'] = (
        d['avgSpeed']
          .replace([np.inf, -np.inf], np.nan)
          .fillna(d['avgSpeed'].median())
    )

    # --- Taxes ---
    if 'baseFare' in d.columns:
        d['taxes']   = d['totalFare'] - d['baseFare']
        d['taxRate'] = (d['taxes'] / d['totalFare']).clip(0, 1)

    # --- Scarcity ---
    d['isLowAvailability'] = (d['seatsRemaining'] <= 3).astype(int)

    # --- Booleans to int ---
    for col in ['isBasicEconomy', 'isRefundable', 'isNonStop']:
        if col in d.columns:
            d[col] = d[col].astype(int)

    return d

print('engineer_features() defined.')

In [ ]:
t0 = time.time()
print('Applying feature engineering...')

data = engineer_features(df_raw)
data.drop_duplicates(keep='first', inplace=True)
data.reset_index(drop=True, inplace=True)

print(f'Done in {time.time()-t0:.1f}s')
print(f'Shape: {data.shape}  |  New columns: {data.shape[1] - df_raw.shape[1]}')

new_cols = [
    'daysFrom', 'flightMonth', 'dayOfWeek', 'isWeekend', 'bookingWindowBucket',
    'travelDurationHours', 'numStops', 'primaryAirline', 'cabinCode',
    'timeOfDay', 'avgSpeed', 'routeNumAirlines', 'isPeakSeason'
]
data[new_cols].head()

## 3. Model Preparation

In [ ]:
NUMERIC_FEATURES = [
    'travelDurationHours', 'totalTravelDistance', 'daysFrom',
    'seatsRemaining', 'numStops', 'numSegments', 'flightMonth',
    'dayOfWeek', 'flightQuarter', 'isWeekend', 'isBasicEconomy',
    'isRefundable', 'isNonStop', 'routePopularity', 'routeNumAirlines',
    'avgSpeed', 'isPeakSeason', 'isHolidaySeason', 'isLowAvailability',
    'numAirlines',
]
if 'taxRate' in data.columns:
    NUMERIC_FEATURES.append('taxRate')

CATEGORICAL_FEATURES = [
    'startingAirport', 'destinationAirport', 'primaryAirline',
    'flightMonthName', 'dayOfWeekName', 'bookingWindowBucket',
    'cabinCode', 'timeOfDay', 'aircraftManufacturer', 'fareClass',
]

ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

y_raw = data[TARGET].values.astype(np.float32)
y     = np.log1p(y_raw)   # log-transform target

print(f'Features: {len(ALL_FEATURES)}  (numeric={len(NUMERIC_FEATURES)}, categorical={len(CATEGORICAL_FEATURES)})')
print(f'Samples:  {len(y):,}')
print(f'Target  — mean=${y_raw.mean():.2f}  median=${np.median(y_raw):.2f}  std=${y_raw.std():.2f}')
print(f'          min=${y_raw.min():.2f}  max=${y_raw.max():.2f}')

In [ ]:
# Label-encode categoricals for LightGBM / XGBoost
X_df = data[ALL_FEATURES].copy()
label_encoders = {}
for col in CATEGORICAL_FEATURES:
    le = LabelEncoder()
    X_df[col] = le.fit_transform(X_df[col].astype(str))
    label_encoders[col] = le

X = X_df.values.astype(np.float32)

# Raw string categoricals for CatBoost (no OHE needed)
X_cat = data[ALL_FEATURES].copy()
for col in CATEGORICAL_FEATURES:
    X_cat[col] = X_cat[col].astype(str)
cat_indices = [ALL_FEATURES.index(c) for c in CATEGORICAL_FEATURES]

print(f'X shape: {X.shape}')
print(f'CatBoost categorical indices: {cat_indices}')

In [ ]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

def fold_metrics(y_true_log, y_pred_log, model_name, fold_num):
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)
    return {
        'Model': model_name,
        'Fold':  fold_num,
        'R2_log':  r2_score(y_true_log, y_pred_log),
        'R2':      r2_score(y_true, y_pred),
        'MAE':     mean_absolute_error(y_true, y_pred),
        'RMSE':    float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'MAPE':    mean_absolute_percentage_error(y_true, y_pred) * 100,
    }

all_results = []   # collects fold dicts from all models
print(f'{N_FOLDS}-fold CV ready. Samples per fold ~{len(y)//N_FOLDS:,}')

## 4. LightGBM — 5-Fold CV

In [ ]:
lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'num_leaves': 127,
    'learning_rate': 0.05,
    'n_estimators': 3000,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': SEED,
    'n_jobs': -1,
    'verbose': -1,
}

lgb_oof  = np.zeros(len(y))
lgb_models = []
lgb_results = []
feature_imp_lgb = np.zeros(len(ALL_FEATURES))

print(f'Training LightGBM ({N_FOLDS}-fold CV)...')
print('-' * 65)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
    t0 = time.time()
    X_tr,  X_val  = X[tr_idx], X[val_idx]
    y_tr,  y_val  = y[tr_idx], y[val_idx]

    m = lgb.LGBMRegressor(**lgb_params)
    m.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(150, verbose=False),
            lgb.log_evaluation(False)
        ],
    )
    preds = m.predict(X_val)
    lgb_oof[val_idx] = preds
    lgb_models.append(m)
    feature_imp_lgb += m.feature_importances_ / N_FOLDS

    row = fold_metrics(y_val, preds, 'LightGBM', fold)
    lgb_results.append(row)
    all_results.append(row)
    print(f'  Fold {fold} | R²={row["R2"]:.4f} | MAE=${row["MAE"]:.2f} '
          f'| RMSE=${row["RMSE"]:.2f} | MAPE={row["MAPE"]:.2f}% '
          f'| iters={m.best_iteration_} | {time.time()-t0:.0f}s')

lgb_df = pd.DataFrame(lgb_results)
print('=' * 65)
print('LightGBM Summary:')
for metric in ['R2', 'MAE', 'RMSE', 'MAPE']:
    print(f'  {metric}: {lgb_df[metric].mean():.4f} ± {lgb_df[metric].std():.4f}')

## 5. CatBoost — 5-Fold CV
CatBoost handles categorical features natively (no label encoding needed).

In [ ]:
cat_params = {
    'iterations': 3000,
    'learning_rate': 0.05,
    'depth': 8,
    'l2_leaf_reg': 3.0,
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'random_seed': SEED,
    'verbose': False,
    'allow_writing_files': False,
    'early_stopping_rounds': 150,
}

cat_oof  = np.zeros(len(y))
cat_models = []
cat_results = []

print(f'Training CatBoost ({N_FOLDS}-fold CV)...')
print('-' * 65)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
    t0 = time.time()
    X_tr  = X_cat.iloc[tr_idx]
    X_val = X_cat.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    train_pool = Pool(X_tr,  y_tr,  cat_features=cat_indices)
    val_pool   = Pool(X_val, y_val, cat_features=cat_indices)

    m = CatBoostRegressor(**cat_params)
    m.fit(train_pool, eval_set=val_pool)

    preds = m.predict(X_val)
    cat_oof[val_idx] = preds
    cat_models.append(m)

    row = fold_metrics(y_val, preds, 'CatBoost', fold)
    cat_results.append(row)
    all_results.append(row)
    print(f'  Fold {fold} | R²={row["R2"]:.4f} | MAE=${row["MAE"]:.2f} '
          f'| RMSE=${row["RMSE"]:.2f} | MAPE={row["MAPE"]:.2f}% '
          f'| iters={m.best_iteration_} | {time.time()-t0:.0f}s')

cat_df = pd.DataFrame(cat_results)
print('=' * 65)
print('CatBoost Summary:')
for metric in ['R2', 'MAE', 'RMSE', 'MAPE']:
    print(f'  {metric}: {cat_df[metric].mean():.4f} ± {cat_df[metric].std():.4f}')

## 6. XGBoost — 5-Fold CV

In [ ]:
xgb_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'n_estimators': 3000,
    'max_depth': 8,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'random_state': SEED,
    'n_jobs': -1,
    'tree_method': 'hist',
}

xgb_oof  = np.zeros(len(y))
xgb_models = []
xgb_results = []

print(f'Training XGBoost ({N_FOLDS}-fold CV)...')
print('-' * 65)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
    t0 = time.time()
    X_tr, X_val = X[tr_idx], X[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    m = xgb.XGBRegressor(**xgb_params)
    m.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False,
        early_stopping_rounds=150,
    )
    preds = m.predict(X_val)
    xgb_oof[val_idx] = preds
    xgb_models.append(m)

    row = fold_metrics(y_val, preds, 'XGBoost', fold)
    xgb_results.append(row)
    all_results.append(row)
    print(f'  Fold {fold} | R²={row["R2"]:.4f} | MAE=${row["MAE"]:.2f} '
          f'| RMSE=${row["RMSE"]:.2f} | MAPE={row["MAPE"]:.2f}% '
          f'| iters={m.best_iteration} | {time.time()-t0:.0f}s')

xgb_df = pd.DataFrame(xgb_results)
print('=' * 65)
print('XGBoost Summary:')
for metric in ['R2', 'MAE', 'RMSE', 'MAPE']:
    print(f'  {metric}: {xgb_df[metric].mean():.4f} ± {xgb_df[metric].std():.4f}')

## 7. Quantile Regression — Prediction Intervals
Predicting the 10th, 50th, and 90th percentiles gives an 80% prediction interval for each flight. This is more honest and useful for consumers than a single point estimate.

In [ ]:
QUANTILES = [0.10, 0.50, 0.90]

q_base = {
    'num_leaves': 127,
    'learning_rate': 0.05,
    'n_estimators': 1500,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'random_state': SEED,
    'n_jobs': -1,
    'verbose': -1,
}

X_qtr, X_qval, y_qtr, y_qval = train_test_split(
    X, y, test_size=0.20, random_state=SEED
)
y_qval_orig = np.expm1(y_qval)

quantile_models = {}
print('Training quantile LightGBM models...')
for q in QUANTILES:
    params = {**q_base, 'objective': 'quantile', 'alpha': q, 'metric': 'quantile'}
    m = lgb.LGBMRegressor(**params)
    m.fit(
        X_qtr, y_qtr,
        eval_set=[(X_qval, y_qval)],
        callbacks=[
            lgb.early_stopping(80, verbose=False),
            lgb.log_evaluation(False)
        ],
    )
    quantile_models[q] = m
    print(f'  Q{q:.0%} trained | best_iter={m.best_iteration_}')

preds_q = {q: np.expm1(quantile_models[q].predict(X_qval)) for q in QUANTILES}

in_interval = (y_qval_orig >= preds_q[0.10]) & (y_qval_orig <= preds_q[0.90])
coverage   = in_interval.mean() * 100
avg_width  = (preds_q[0.90] - preds_q[0.10]).mean()

print(f'\nInterval statistics [P10–P90]:')
print(f'  Coverage: {coverage:.1f}%  (ideal ≈ 80%)')
print(f'  Avg width: ${avg_width:.2f}')

In [ ]:
samp_n = 2000
samp_idx = np.random.choice(len(y_qval_orig), samp_n, replace=False)
sort_idx = np.argsort(y_qval_orig[samp_idx])

y_s   = y_qval_orig[samp_idx][sort_idx]
p10_s = preds_q[0.10][samp_idx][sort_idx]
p50_s = preds_q[0.50][samp_idx][sort_idx]
p90_s = preds_q[0.90][samp_idx][sort_idx]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

ax1.fill_between(range(samp_n), p10_s, p90_s, alpha=0.3,
                 color='steelblue', label='80% Interval [P10–P90]')
ax1.plot(p50_s, color='steelblue', lw=1.5, label='Median prediction (P50)')
ax1.scatter(range(samp_n), y_s, color='red', s=1, alpha=0.4, label='True price')
ax1.set_xlabel('Sample (sorted by true price)')
ax1.set_ylabel('Flight price ($)')
ax1.set_title(
    f'Quantile Prediction Intervals\nCoverage={coverage:.1f}%  Avg Width=${avg_width:.0f}',
    fontweight='bold'
)
ax1.legend()

ax2.scatter(y_s, p90_s - p10_s, s=2, alpha=0.4, color='steelblue')
ax2.set_xlabel('True Price ($)')
ax2.set_ylabel('Interval Width ($)')
ax2.set_title('Prediction Uncertainty vs True Price\n(Wider = more uncertain)', fontweight='bold')
ax2.set_xlim(0, 2000)

plt.suptitle('Quantile Regression — Prediction Intervals', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('quantile_intervals.png', dpi=150, bbox_inches='tight')
plt.show()
print('Note: uncertainty grows with price — reflects heteroscedasticity in airline pricing.')

## 8. TabNet — Attention-Based Deep Learning
TabNet (Arik & Pfister, 2021) applies sequential attention to select features at each decision step. It provides instance-wise feature importances and is competitive with GBDT on tabular data.

We train on a 500k sample (manageable on GPU/CPU) to compare with the GBDT models.

In [ ]:
TABNET_N = min(500_000, len(X))
tn_idx = np.random.choice(len(X), TABNET_N, replace=False)
X_tn = X[tn_idx]
y_tn = y[tn_idx]

X_tn_tr, X_tn_val, y_tn_tr, y_tn_val = train_test_split(
    X_tn, y_tn, test_size=0.20, random_state=SEED
)

tabnet = TabNetRegressor(
    n_d=32, n_a=32, n_steps=5,
    gamma=1.3,
    n_independent=2, n_shared=2,
    momentum=0.02,
    lambda_sparse=1e-4,
    optimizer_fn=torch.optim.Adam,
    optimizer_params={'lr': 2e-3, 'weight_decay': 1e-5},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    scheduler_params={'step_size': 20, 'gamma': 0.9},
    mask_type='entmax',
    seed=SEED,
    verbose=10,
    device_name='cuda' if torch.cuda.is_available() else 'cpu',
)

print(f'Training TabNet on {TABNET_N:,} samples...')
tabnet.fit(
    X_tn_tr, y_tn_tr.reshape(-1, 1),
    eval_set=[(X_tn_val, y_tn_val.reshape(-1, 1))],
    eval_metric=['rmse'],
    max_epochs=200,
    patience=25,
    batch_size=16384,
    virtual_batch_size=2048,
)

y_tn_pred = tabnet.predict(X_tn_val).squeeze()
y_tn_val_orig  = np.expm1(y_tn_val)
y_tn_pred_orig = np.expm1(y_tn_pred)

tn_r2   = r2_score(y_tn_val_orig, y_tn_pred_orig)
tn_mae  = mean_absolute_error(y_tn_val_orig, y_tn_pred_orig)
tn_rmse = float(np.sqrt(mean_squared_error(y_tn_val_orig, y_tn_pred_orig)))
tn_mape = mean_absolute_percentage_error(y_tn_val_orig, y_tn_pred_orig) * 100

print(f'\nTabNet (trained on {TABNET_N:,} samples):')
print(f'  R²={tn_r2:.4f} | MAE=${tn_mae:.2f} | RMSE=${tn_rmse:.2f} | MAPE={tn_mape:.2f}%')
tn_importance = tabnet.feature_importances_

## 9. Model Comparison

In [ ]:
summary = pd.DataFrame([
    {
        'Model': 'Linear Regression (baseline)',
        'R²': 0.6382,
        'MAE ($)': float('nan'),
        'RMSE ($)': float('nan'),
        'MAPE (%)': 4.85,
        'CV Folds': 1,
        'Note': 'log(Y), OHE + interaction terms'
    },
    {
        'Model': 'LightGBM',
        'R²': lgb_df['R2'].mean(),
        'MAE ($)': lgb_df['MAE'].mean(),
        'RMSE ($)': lgb_df['RMSE'].mean(),
        'MAPE (%)': lgb_df['MAPE'].mean(),
        'CV Folds': N_FOLDS,
        'Note': f'± R²={lgb_df["R2"].std():.4f}'
    },
    {
        'Model': 'CatBoost',
        'R²': cat_df['R2'].mean(),
        'MAE ($)': cat_df['MAE'].mean(),
        'RMSE ($)': cat_df['RMSE'].mean(),
        'MAPE (%)': cat_df['MAPE'].mean(),
        'CV Folds': N_FOLDS,
        'Note': f'± R²={cat_df["R2"].std():.4f}'
    },
    {
        'Model': 'XGBoost',
        'R²': xgb_df['R2'].mean(),
        'MAE ($)': xgb_df['MAE'].mean(),
        'RMSE ($)': xgb_df['RMSE'].mean(),
        'MAPE (%)': xgb_df['MAPE'].mean(),
        'CV Folds': N_FOLDS,
        'Note': f'± R²={xgb_df["R2"].std():.4f}'
    },
    {
        'Model': 'TabNet',
        'R²': tn_r2,
        'MAE ($)': tn_mae,
        'RMSE ($)': tn_rmse,
        'MAPE (%)': tn_mape,
        'CV Folds': 1,
        'Note': f'{TABNET_N//1000}k sample'
    },
])

print('=' * 75)
print('MODEL COMPARISON')
print('=' * 75)
print(summary.to_string(index=False))
print('=' * 75)

In [ ]:
models  = ['Linear\nRegression', 'LightGBM', 'CatBoost', 'XGBoost', 'TabNet']
colors  = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12', '#9b59b6']
r2_vals = [0.6382, lgb_df['R2'].mean(), cat_df['R2'].mean(),
           xgb_df['R2'].mean(), tn_r2]
mae_vals  = [float('nan'), lgb_df['MAE'].mean(), cat_df['MAE'].mean(),
             xgb_df['MAE'].mean(), tn_mae]
mape_vals = [4.85, lgb_df['MAPE'].mean(), cat_df['MAPE'].mean(),
             xgb_df['MAPE'].mean(), tn_mape]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, vals, ylabel, title in [
    (axes[0], r2_vals,   'R²',       'R² Score (higher = better)'),
    (axes[1], mae_vals,  'MAE ($)',   'MAE in $ (lower = better)'),
    (axes[2], mape_vals, 'MAPE (%)', 'MAPE % (lower = better)'),
]:
    bars = ax.bar(models, vals, color=colors, edgecolor='black', linewidth=0.8)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(ylabel)
    for bar, val in zip(bars, vals):
        if not (val != val):   # skip NaN
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.01,
                f'{val:.3f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold'
            )

plt.suptitle('Model Performance Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# CV fold score distribution
cv_data = pd.concat([
    lgb_df[['Model', 'R2', 'MAE', 'MAPE']],
    cat_df[['Model', 'R2', 'MAE', 'MAPE']],
    xgb_df[['Model', 'R2', 'MAE', 'MAPE']],
])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col, label in [
    (axes[0], 'R2',   'R² per fold'),
    (axes[1], 'MAE',  'MAE ($) per fold'),
    (axes[2], 'MAPE', 'MAPE (%) per fold'),
]:
    cv_data.boxplot(column=col, by='Model', ax=ax)
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('')

plt.suptitle('Cross-Validation Score Distribution (5 folds)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cv_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Statistical significance: Wilcoxon signed-rank test (paired on fold MAPE)
print('Wilcoxon Signed-Rank Tests  |  H0: no difference  |  alpha=0.05')
print('-' * 60)

ref_mapes = lgb_df['MAPE'].values
for name, df_other in [('CatBoost', cat_df), ('XGBoost', xgb_df)]:
    other_mapes = df_other['MAPE'].values
    stat, pval  = wilcoxon(ref_mapes, other_mapes)
    sig = 'SIGNIFICANT *' if pval < 0.05 else 'not significant'
    better = 'LightGBM' if ref_mapes.mean() < other_mapes.mean() else name
    print(f'  LightGBM vs {name:8s} | p={pval:.4f} | {sig} | Better: {better}')

## 10. SHAP Explainability
We use SHAP TreeExplainer on the best LightGBM fold. SHAP values show *how much* each feature pushes the prediction up or down for every sample — much more informative than global feature importance.

In [ ]:
best_fold = lgb_df['R2'].idxmax()
best_lgb  = lgb_models[best_fold]

SHAP_N = min(50_000, len(X))
shap_idx = np.random.choice(len(X), SHAP_N, replace=False)
X_shap   = X[shap_idx]

print(f'Computing SHAP values on {SHAP_N:,} samples (best LightGBM fold {best_fold+1})...')
explainer   = shap.TreeExplainer(best_lgb)
shap_values = explainer.shap_values(X_shap)
print('Done.')

In [ ]:
plt.figure(figsize=(12, 9))
shap.summary_plot(
    shap_values, X_shap,
    feature_names=ALL_FEATURES,
    max_display=20,
    show=False
)
plt.title('SHAP Feature Impact — Flight Price Prediction\n(red=high value, blue=low value)',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
shap.summary_plot(
    shap_values, X_shap,
    feature_names=ALL_FEATURES,
    plot_type='bar',
    max_display=20,
    show=False
)
plt.title('Mean |SHAP Value| — Global Feature Importance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Dependence plots for top-4 features by mean |SHAP|
top4_idx = np.argsort(np.abs(shap_values).mean(axis=0))[::-1][:4]
top4_names = [ALL_FEATURES[i] for i in top4_idx]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, feat_idx, feat_name in zip(axes.flatten(), top4_idx, top4_names):
    shap.dependence_plot(
        feat_idx, shap_values, X_shap,
        feature_names=ALL_FEATURES,
        ax=ax, show=False
    )
    ax.set_title(f'SHAP Dependence: {feat_name}', fontweight='bold')

plt.suptitle('SHAP Dependence Plots — Top 4 Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Feature Importance: GBDT vs TabNet

In [ ]:
fi_lgb = pd.DataFrame({'Feature': ALL_FEATURES, 'LightGBM': feature_imp_lgb})
fi_tn  = pd.DataFrame({'Feature': ALL_FEATURES, 'TabNet':   tn_importance})
fi_all = fi_lgb.merge(fi_tn, on='Feature')

# Normalise to [0,1] for side-by-side comparison
fi_all['LightGBM_norm'] = fi_all['LightGBM'] / fi_all['LightGBM'].max()
fi_all['TabNet_norm']   = fi_all['TabNet']   / fi_all['TabNet'].max()

top20 = fi_all.nlargest(20, 'LightGBM_norm').sort_values('LightGBM_norm')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

ax1.barh(top20['Feature'], top20['LightGBM_norm'],
         color='steelblue', edgecolor='black', linewidth=0.5)
ax1.set_title('LightGBM — Normalised Feature Importance\n(avg over 5 folds)',
              fontweight='bold')
ax1.set_xlabel('Importance (normalised)')
ax1.tick_params(axis='y', labelsize=9)

ax2.barh(top20['Feature'], top20['TabNet_norm'],
         color='mediumpurple', edgecolor='black', linewidth=0.5)
ax2.set_title('TabNet — Normalised Feature Importance\n(attention mask aggregation)',
              fontweight='bold')
ax2.set_xlabel('Importance (normalised)')
ax2.tick_params(axis='y', labelsize=9)

plt.suptitle('Feature Importance: GBDT vs Deep Learning', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Error Analysis
Using LightGBM out-of-fold predictions (OOF) — every sample is predicted exactly once on data it was not trained on, making this an unbiased evaluation.

In [ ]:
y_pred_oof  = np.expm1(lgb_oof)
y_true_orig = np.expm1(y)
errors      = y_pred_oof - y_true_orig
abs_errors  = np.abs(errors)
pct_errors  = abs_errors / y_true_orig * 100

print('LightGBM OOF Error Summary:')
print(f'  Bias (mean error): ${errors.mean():.2f}  (+ = overestimate)')
print(f'  MAE:               ${abs_errors.mean():.2f}')
print(f'  RMSE:              ${np.sqrt((errors**2).mean()):.2f}')
print(f'  MAPE:              {pct_errors.mean():.2f}%')
print(f'  Within $25:        {(abs_errors <= 25).mean()*100:.1f}%')
print(f'  Within $50:        {(abs_errors <= 50).mean()*100:.1f}%')
print(f'  Within $100:       {(abs_errors <= 100).mean()*100:.1f}%')

In [ ]:
samp_e = np.random.choice(len(y_true_orig), 15_000, replace=False)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# --- Error histogram ---
axes[0, 0].hist(errors, bins=150, color='steelblue', edgecolor='white', linewidth=0.2)
axes[0, 0].axvline(0,            color='red',    lw=2, label='Zero error')
axes[0, 0].axvline(errors.mean(), color='orange', lw=2, ls='--',
                   label=f'Mean: ${errors.mean():.1f}')
axes[0, 0].set_xlim(-600, 600)
axes[0, 0].set_xlabel('Prediction error ($)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Error Distribution (LightGBM OOF)', fontweight='bold')
axes[0, 0].legend()

# --- Actual vs Predicted ---
max_price = min(float(y_true_orig.max()), 2500.0)
axes[0, 1].scatter(y_true_orig[samp_e], y_pred_oof[samp_e],
                   s=1, alpha=0.25, color='steelblue')
axes[0, 1].plot([0, max_price], [0, max_price], 'r--', lw=2, label='Perfect prediction')
axes[0, 1].set_xlim(0, max_price)
axes[0, 1].set_ylim(0, max_price)
axes[0, 1].set_xlabel('True price ($)')
axes[0, 1].set_ylabel('Predicted price ($)')
axes[0, 1].set_title('Actual vs Predicted (15k sample)', fontweight='bold')
axes[0, 1].legend()

# --- MAE by price bucket ---
price_buckets = pd.cut(
    y_true_orig,
    bins=[0, 100, 200, 350, 500, 750, 1000, 5000],
    labels=['<$100', '$100-200', '$200-350', '$350-500', '$500-750', '$750-1k', '>$1k']
)
err_bucket = pd.Series(abs_errors).groupby(price_buckets).mean()
axes[1, 0].bar(err_bucket.index, err_bucket.values,
               color='coral', edgecolor='black', lw=0.8)
axes[1, 0].set_xlabel('True price bucket')
axes[1, 0].set_ylabel('Mean absolute error ($)')
axes[1, 0].set_title('MAE by Price Bucket', fontweight='bold')
axes[1, 0].tick_params(axis='x', rotation=30)

# --- Residual plot ---
axes[1, 1].scatter(y_pred_oof[samp_e], errors[samp_e],
                   s=1, alpha=0.25, color='steelblue')
axes[1, 1].axhline(0, color='red', lw=2, ls='--')
axes[1, 1].set_xlim(0, 1800)
axes[1, 1].set_ylim(-600, 600)
axes[1, 1].set_xlabel('Predicted price ($)')
axes[1, 1].set_ylabel('Residual ($)')
axes[1, 1].set_title('Residual Plot — LightGBM OOF', fontweight='bold')

plt.suptitle('Error Analysis — LightGBM (Out-of-Fold)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# MAE breakdown by key categorical features (using LightGBM OOF)
err_df = data[['primaryAirline', 'cabinCode', 'bookingWindowBucket',
               'numStops', 'timeOfDay']].copy()
err_df['abs_error'] = abs_errors
err_df['pct_error'] = pct_errors

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for ax, col, title in [
    (axes[0], 'bookingWindowBucket', 'MAE by Booking Window'),
    (axes[1], 'numStops',            'MAE by Number of Stops'),
    (axes[2], 'timeOfDay',           'MAE by Departure Time'),
]:
    grp = err_df.groupby(col)['abs_error'].mean().sort_values(ascending=False)
    ax.bar(grp.index.astype(str), grp.values, color='steelblue',
           edgecolor='black', lw=0.8)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Mean Absolute Error ($)')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Error Breakdown by Categorical Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('error_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Simple Ensemble (OOF Blending)
Blending the LightGBM, CatBoost, and XGBoost OOF predictions often reduces error further.

In [ ]:
# Equal-weight average of three OOF arrays
ensemble_oof = (lgb_oof + cat_oof + xgb_oof) / 3.0

y_ens_pred = np.expm1(ensemble_oof)
y_ens_true = np.expm1(y)

ens_r2   = r2_score(y_ens_true, y_ens_pred)
ens_mae  = mean_absolute_error(y_ens_true, y_ens_pred)
ens_rmse = float(np.sqrt(mean_squared_error(y_ens_true, y_ens_pred)))
ens_mape = mean_absolute_percentage_error(y_ens_true, y_ens_pred) * 100

print('Ensemble (LightGBM + CatBoost + XGBoost, equal weight):')
print(f'  R²={ens_r2:.4f} | MAE=${ens_mae:.2f} | RMSE=${ens_rmse:.2f} | MAPE={ens_mape:.2f}%')
print()
print('Individual OOF comparison:')
print(f'  LightGBM: R²={r2_score(y_ens_true, np.expm1(lgb_oof)):.4f}')
print(f'  CatBoost: R²={r2_score(y_ens_true, np.expm1(cat_oof)):.4f}')
print(f'  XGBoost:  R²={r2_score(y_ens_true, np.expm1(xgb_oof)):.4f}')
print(f'  Ensemble: R²={ens_r2:.4f}  <-- typically best')

## 14. Conclusions

### Results Summary
| Model | R² | Key Advantage |
|---|---|---|
| Linear Regression (baseline) | 0.64 | Interpretable, fast |
| **LightGBM** | — | Best single GBDT; fast training on 4M rows |
| **CatBoost** | — | Native categorical handling; no OHE needed |
| **XGBoost** | — | Robust; easy GPU acceleration |
| **Ensemble** | — | Typically highest R²; best for production |
| **TabNet** | — | Instance-wise attention; interpretable DL |

### Key Feature Findings (SHAP)
- **daysFrom** (booking lead time) is the dominant driver — last-minute flights spike in price.
- **totalTravelDistance / travelDurationHours** — longer routes cost more, but non-linearly.
- **seatsRemaining** — scarcity effect is real and captured by SHAP.
- **primaryAirline** — carrier identity matters more than aircraft type.
- **bookingWindowBucket** — confirms that booking 2–4 weeks ahead minimises cost.
- **cabinCode** and **isBasicEconomy** — cabin class creates the largest absolute price difference.

### Limitations & Future Work
- Dataset covers only April–October 2022 (no winter / holiday season).
- No real-time competitor prices or fuel cost data.
- Temporal Fusion Transformer (TFT) would better exploit the time-series nature of pricing.
- Hyperparameter tuning via Optuna would further improve GBDT performance.
- FT-Transformer (Gorishniy et al., 2021) is worth benchmarking against TabNet.